In [ ]:
!pip install langchain langchain-community langchain-huggingface pypdf chromadb sentence-transformers langchain-google-genai langchain-google-genai google-generativeai -q langchain-groq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
# Point this to the folder in your Google Drive where you saved the PDFs
loader = PyPDFDirectoryLoader("/content/drive/MyDrive/Knowledge Base")
docs = loader.load()

/tmp/ipykernel_1012/2529997866.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# Ensure we have chunks before proceeding
if not chunks:
    raise ValueError("The 'chunks' list is empty. Please verify previous PDF loading steps!")

print("⚡ Building vector database using local HuggingFace embeddings...")

# 1. Initialize local HuggingFace embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create, store, and persist the vectors in the Chroma database folder
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("✅ Vector database successfully created and saved to './chroma_db'!")


⚡ Building vector database using local HuggingFace embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector database successfully created and saved to './chroma_db'!


In [ ]:
# 1. Install dependencies
!pip install -q -U langchain-groq

import os
import json
from google.colab import userdata
from langchain_groq import ChatGroq
from IPython.display import Markdown, display

# 2. Initialize the LLM (key comes from Colab Secrets, never hardcode it)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

MODEL_NAME = "openai/gpt-oss-120b"   # alternative: "qwen/qwen3.6-27b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    timeout=60,
    max_retries=2,
)

# 3. Office roles and sheets
OFFICE_ROLES = {
    "CPDO": {
        "name": "City Planning and Development Office",
        "sheets": ['Summary statistics', 'Population', 'Building', 'SDG']
    },
    "SWO": {
        "name": "City Social Welfare and Development Office",
        "sheets": ['Food insecurity experience', 'Key employment statistics',
                   'Informal employment', 'Overcrowding status', 'Financial account']
    },
    "DRRMO": {
        "name": "Disaster Risk Reduction and Management Office",
        "sheets": ['DRRM', 'Roof material', 'Outer walls material', 'Floor material']
    }
}

# 4. Proactive scan function
def proactive_barangay_scan(barangay_name):
    if barangay_name not in df.index:
        return f"❌ Barangay '{barangay_name}' not found."

    bgy_row = df.loc[barangay_name]

    # Simple trigger logic
    triggered_office = "CPDO"
    detected_crisis = "General infrastructure and development backlog based on composite CBMS indicators."

    for col, val in bgy_row.items():
        col_lower = str(col).lower()
        try:
            if ('vulnerability_index' in col_lower and val > 70) or ('4ps' in col_lower and val > 50):
                triggered_office = "SWO"
                detected_crisis = f"Critical high vulnerability or social welfare dependency flag detected ({col}: {val})."
                break
            elif 'roof material' in col_lower and val > 40:
                triggered_office = "DRRMO"
                detected_crisis = f"High concentration of vulnerable housing materials detected ({col}: {val})."
                break
        except TypeError:
            continue  # skip non-numeric columns

    role_info = OFFICE_ROLES.get(triggered_office, OFFICE_ROLES["CPDO"])

    # Filter columns to keep the prompt light and fast
    allowed_cols = [
        col for col in df.columns
        if any(s.lower() in str(col).lower() for s in role_info["sheets"]) or col == 'Vulnerability_Index'
    ]
    bgy_filtered = bgy_row[allowed_cols].head(15).round(2).to_dict()

    # Retrieve local RAG context if vector_db is available
    legal_context = "Republic Act No. 7160 (Local Government Code), RA 10121 (DRRM Act), RA 11315 (CBMS Act)."
    if 'vector_db' in globals():
        try:
            retrieved_docs = vector_db.similarity_search(
                f"{role_info['name']} {detected_crisis}", k=1
            )
            legal_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
        except Exception:
            pass

    prompt = f"""
    You are the Proactive Policy Engine for Project Banwaan, acting for the {role_info['name']} ({triggered_office}) in Naga City.

    AUTOMATED TRIGGER: Barangay {barangay_name.upper()} flagged due to: "{detected_crisis}".

    KEY CBMS METRICS:
    {json.dumps(bgy_filtered, indent=2, default=str)}

    LEGAL BASIS / RAG:
    {legal_context}

    TASK: Proactively generate a policy and program proposal tailored for the Barangay Captain of {barangay_name.upper()}.

    STRICT FORMAT REQUIREMENTS:

    ### 1. Overview
    (Provide a concise summary of the barangay's current situation based on CBMS metrics. **CRITICAL: Must be strictly a maximum of 150 words.**)

    ### 2. Policy
    (Provide exactly 5 distinct, actionable policies, programs, or local ordinances. Number them 1 to 5. Cite specific Republic Acts and Sections where applicable.)

    ### 3. SDG Alignment
    (Detail at least 3 Sustainable Development Goals, such as SDG 1, SDG 8, SDG 9, or SDG 11, that these policies fulfill, explaining how each goal is addressed.)

    ### 4. Impact
    (Detail the expected changes in indicators, estimated cost/budget source, and targeted improvement for the barangay.)
    """

    response = llm.invoke(prompt)

    # Handle both string and list-of-parts responses
    if isinstance(response.content, list):
        return "".join(
            p.get("text", "") if isinstance(p, dict) else str(p)
            for p in response.content
        )
    return response.content

print(f"✅ Groq engine initialized with {MODEL_NAME}")

# 5. Run the scan on the top barangay
top_bgy = df.index[0]
print(f"🚨 MAP ALERT TRIGGERED FOR: {top_bgy}\n" + "=" * 50)

proactive_output = proactive_barangay_scan(top_bgy)
display(Markdown(proactive_output))

✅ Groq engine initialized with openai/gpt-oss-120b
🚨 MAP ALERT TRIGGERED FOR: Barangay 9


### 1. Overview  
Barangay 9 of Naga City records **5,706 housing units**, of which **89.82 % (≈5,125 units)** have roofs made of highly vulnerable materials (e.g., nipa, cogon, thatch). Only 4.85 % use metal sheets and 3.12 % employ more resilient options. This concentration elevates exposure to typhoons, heavy rain, and fire, jeopardizing lives, livelihoods, and the barangay’s overall disaster‑risk profile. Immediate, coordinated interventions are required to retrofit roofs, secure financing, and embed resilient housing standards into local planning.

### 2. Policy  
1. **Barangay Ordinance on Roof Retrofit & Replacement** – Mandates upgrading all vulnerable roofs to fire‑ and wind‑resistant materials within three years. *Cites*: RA 11058 §101 (building code), RA 10121 §3 (DRRM).  
2. **Barangay Housing Resilience Fund** – Sets aside 5 % of the Barangay’s IRA, matched by a grant from the National Housing Authority (NHA) under RA 7279 §5, to subsidize retrofits for low‑income families.  
3. **Community‑Based Roof Material Survey & GIS Mapping** – Institutionalizes an annual CBMS‑driven mapping of roof types for risk assessment and planning. *Cites*: RA 10121 §14; RA 7160 §371 (barangay development plans).  
4. **Public Rental Housing Partnership** – Collaborates with NHA and the City LGU to construct 50 socialized rental units built to resilient standards (metal roofing, reinforced walls). *Cites*: RA 7279 §2; RA 11058 §103.  
5. **Capacity‑Building & Early‑Warning Training** – Conducts quarterly DRRM and safe‑construction workshops for residents, integrating local early‑warning protocols. *Cites*: RA 10121 §5 (DRRM Council), DOH Community Health Guidelines.

### 3. SDG Alignment  
- **SDG 1 – No Poverty**: Financial subsidies and public rental units reduce housing cost burdens for vulnerable households.  
- **SDG 11 – Sustainable Cities and Communities**: Upgrading roofs and expanding resilient housing improve safety, livability, and inclusive urban development.  
- **SDG 13 – Climate Action**: Strengthening structures against extreme weather mitigates climate‑related risks and enhances community adaptive capacity.  

### 4. Impact  
- **Indicator Changes**: Reduce vulnerable‑roof share from 89.8 % to ≤30 % within five years; increase “safe housing” proportion to ≥70 %; lower projected disaster loss value by ~40 %.  
- **Estimated Cost & Budget Sources**: Roof retrofits – ₱15,000 × 5,000 units = ₱75 M; Public rental units – ₱1.2 B (NHA grant 60 % + Barangay IRA 20 % + CSR 20 %). Funding via IRA allocation, NHA matching grant, National DRRM Fund, and private sector CSR.  
- **Targeted Improvement**: Enhanced household resilience index, reduced casualty risk in typhoon/flood events, and strengthened compliance with national building and DRRM statutes.

In [ ]:
import pandas as pd
import numpy as np

file_name = 'Anonymized-Data_Team-Synapse.xlsx'
xls = pd.ExcelFile(file_name)

all_sheets_data = []

for sheet in xls.sheet_names:
    # Read without headers so the row indices match exactly
    raw_sheet = pd.read_excel(file_name, sheet_name=sheet, header=None)

    # Find the row where 'Barangay' appears in ANY column
    header_idx = None
    for idx, row in raw_sheet.iterrows():
        if row.astype(str).str.contains('Barangay', case=False).any():
            header_idx = idx
            break

    if header_idx is not None:
        # Skip rows up to the header index
        df_sheet = pd.read_excel(file_name, sheet_name=sheet, skiprows=header_idx)

        # Identify which column actually contains the 'Barangay' names
        bgy_col = None
        for col in df_sheet.columns:
            if 'barangay' in str(col).lower():
                bgy_col = col
                break

        if bgy_col:
            # Clean up the Barangay names
            df_sheet[bgy_col] = df_sheet[bgy_col].astype(str).str.strip()

            # Keep only rows that are actual Barangays (e.g., "Barangay 1" to "Barangay 27")
            df_bgy = df_sheet[df_sheet[bgy_col].str.startswith('Barangay ', na=False)].copy()

            # Only merge sheets that have Barangay-level rows
            if len(df_bgy) >= 10:
                df_bgy.set_index(bgy_col, inplace=True)
                df_bgy.index.name = 'Barangay'

                # Convert numeric columns properly
                for col in df_bgy.columns:
                    df_bgy[col] = pd.to_numeric(df_bgy[col], errors='coerce')
                df_bgy = df_bgy.select_dtypes(include=[np.number]).fillna(0)

                # Add the Sheet Name to every column so the AI knows exact context!
                df_bgy.columns = [f"[{sheet}] {col}" for col in df_bgy.columns]
                all_sheets_data.append(df_bgy)

# Combine all Barangay sheets into one Master DataFrame
if all_sheets_data:
    df = pd.concat(all_sheets_data, axis=1)
    # Use .copy() to defragment the DataFrame after a massive concatenation
    df = df.loc[:, ~df.columns.duplicated()].fillna(0).copy()

    # Calculate the 0-100 Vulnerability Index across all sheets for the 3D Map
    df_norm = df.apply(lambda x: (x - x.min()) / (x.max() - x.min()) * 100 if (x.max() - x.min()) != 0 else pd.Series(0, index=x.index))
    if isinstance(df_norm, pd.DataFrame):
        df['Vulnerability_Index'] = df_norm.mean(axis=1).round(2)
    else:
        df['Vulnerability_Index'] = df_norm.round(2)
    df.sort_values(by='Vulnerability_Index', ascending=False, inplace=True)

    # Export for the 3D Map Team
    df.to_csv('Banwaan_Full_Dataset_Map_And_AI.csv')
    print(f"✅ Merged {len(all_sheets_data)} Barangay-level sheets into {len(df.columns)} total columns across {len(df)} Barangays!")
    display(df[['Vulnerability_Index']].head())
else:
    print("❌ Could not find any valid Barangay data across the sheets.")

✅ Merged 59 Barangay-level sheets into 567 total columns across 27 Barangays!


,Vulnerability_Index
Barangay,
Barangay 9,51.64
Barangay 21,42.38
Barangay 6,38.76
Barangay 17,37.23
Barangay 4,33.19


In [ ]:
# 1. Install dependencies
!pip install -q -U langchain-groq

import os
import json
from google.colab import userdata
from langchain_groq import ChatGroq
from IPython.display import Markdown, display

# 2. Initialize the LLM (key comes from Colab Secrets, never hardcode it)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

MODEL_NAME = "openai/gpt-oss-120b"   # alternative: "qwen/qwen3.6-27b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    timeout=60,
    max_retries=2,
)

# 3. Office roles and sheets
OFFICE_ROLES = {
    "CPDO": {
        "name": "City Planning and Development Office",
        "sheets": ['Summary statistics', 'Population', 'Building', 'SDG']
    },
    "SWO": {
        "name": "City Social Welfare and Development Office",
        "sheets": ['Food insecurity experience', 'Key employment statistics',
                   'Informal employment', 'Overcrowding status', 'Financial account']
    },
    "DRRMO": {
        "name": "Disaster Risk Reduction and Management Office",
        "sheets": ['DRRM', 'Roof material', 'Outer walls material', 'Floor material']
    }
}

# 4. Proactive scan function
def proactive_barangay_scan(barangay_name):
    if barangay_name not in df.index:
        return f"❌ Barangay '{barangay_name}' not found."

    bgy_row = df.loc[barangay_name]

    # Simple trigger logic
    triggered_office = "CPDO"
    detected_crisis = "General infrastructure and development backlog based on composite CBMS indicators."

    for col, val in bgy_row.items():
        col_lower = str(col).lower()
        try:
            if ('vulnerability_index' in col_lower and val > 70) or ('4ps' in col_lower and val > 50):
                triggered_office = "SWO"
                detected_crisis = f"Critical high vulnerability or social welfare dependency flag detected ({col}: {val})."
                break
            elif 'roof material' in col_lower and val > 40:
                triggered_office = "DRRMO"
                detected_crisis = f"High concentration of vulnerable housing materials detected ({col}: {val})."
                break
        except TypeError:
            continue  # skip non-numeric columns

    role_info = OFFICE_ROLES.get(triggered_office, OFFICE_ROLES["CPDO"])

    # Filter columns to keep the prompt light and fast
    allowed_cols = [
        col for col in df.columns
        if any(s.lower() in str(col).lower() for s in role_info["sheets"]) or col == 'Vulnerability_Index'
    ]
    bgy_filtered = bgy_row[allowed_cols].head(15).round(2).to_dict()

    # Retrieve local RAG context if vector_db is available
    legal_context = "Republic Act No. 7160 (Local Government Code), RA 10121 (DRRM Act), RA 11315 (CBMS Act)."
    if 'vector_db' in globals():
        try:
            retrieved_docs = vector_db.similarity_search(
                f"{role_info['name']} {detected_crisis}", k=1
            )
            legal_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
        except Exception:
            pass

    prompt = f"""
    You are the Proactive Policy Engine for Project Banwaan, acting for the {role_info['name']} ({triggered_office}) in Naga City.

    AUTOMATED TRIGGER: Barangay {barangay_name.upper()} flagged due to: "{detected_crisis}".

    KEY CBMS METRICS:
    {json.dumps(bgy_filtered, indent=2, default=str)}

    LEGAL BASIS / RAG:
    {legal_context}

    TASK: Proactively generate a policy and program proposal tailored for the Barangay Captain of {barangay_name.upper()}.

    STRICT FORMAT REQUIREMENTS:

    ### 1. Overview
    (Provide a concise summary of the barangay's current situation based on CBMS metrics. **CRITICAL: Must be strictly a maximum of 150 words.**)

    ### 2. Policy
    (Provide exactly 5 distinct, actionable policies, programs, or local ordinances. Number them 1 to 5. Cite specific Republic Acts and Sections where applicable. Try to have catchy and witty names for the projects like, bantaybata tungod mo linig mo program)

    ### 3. SDG Alignment
    (Detail at least 3 Sustainable Development Goals, such as SDG 1, SDG 8, SDG 9, or SDG 11, that these policies fulfill, explaining how each goal is addressed.)

    ### 4. Impact
    (Detail the expected changes in indicators, estimated cost/budget source, and targeted improvement for the barangay.)
    """

    response = llm.invoke(prompt)

    # Handle both string and list-of-parts responses
    if isinstance(response.content, list):
        return "".join(
            p.get("text", "") if isinstance(p, dict) else str(p)
            for p in response.content
        )
    return response.content

print(f"✅ Groq engine initialized with {MODEL_NAME}")

# 5. Run the scan on the top barangay
top_bgy = df.index[0]
print(f"🚨 MAP ALERT TRIGGERED FOR: {top_bgy}\n" + "=" * 50)

proactive_output = proactive_barangay_scan(top_bgy)
display(Markdown(proactive_output))

✅ Groq engine initialized with openai/gpt-oss-120b
🚨 MAP ALERT TRIGGERED FOR: Barangay 9


### 1. Overview  
Barangay 9 records 5,706 households, of which 5,125 (89.8 %) use highly vulnerable roofing materials (e.g., nipa, cogon, sheet metal without proper anchorage). The remaining 4.85 % are tiled, 3.12 % are metal‑sheet with proper fastening, and the rest are negligible. This exposure markedly raises the barangay’s risk to wind, flood, and fire hazards, threatening lives, livelihoods, and public health. Immediate DRRM‑linked housing interventions are required to lower the vulnerability index and comply with national disaster‑risk reduction mandates.

### 2. Policy  

1. **BantayKubo Safe‑Roof Initiative** – Barangay ordinance mandating that all new and replacement roofs meet the standards of **RA 11058 (National Building Code), Sec. 1** and **RA 10533 (DRRM Act), Sec. 3(b)**. Grants of ₱15,000 per unit for low‑income families to upgrade to reinforced concrete or steel‑truss roofs.  

2. **LigtasBaha Flood‑Resilient Housing Ordinance** – Adoption of **RA 7160 (Local Government Code), Sec. 379** to require elevation of dwellings in flood‑prone zones and installation of rain‑water harvesting systems. Funding through the **National Disaster Risk Reduction and Management Fund (NDRRMF)**.  

3. **KapitBahay Micro‑Finance Housing Grant** – Partnership with the **National Housing Authority (RA 10630, Sec. 2)** to provide interest‑free micro‑loans of up to ₱200,000 for roof retrofitting, prioritized for households in the “Frequency.1” (277) and “Frequency.2” (178) categories.  

4. **GreenRoof Barangay 9 Sustainable Shelter Program** – Incentivize use of eco‑friendly, fire‑resistant roofing (e.g., fiber‑cement, solar‑panel tiles) through tax rebates under **RA 10963 (Tax Reform for Acc. Housing), Sec. 84**.  

5. **Barangay 9 Disaster‑Ready Housing Inspection Taskforce** – Creation of a barangay‑level taskforce under **RA 10533, Sec. 5** to conduct annual roof safety audits, issue “Safe‑Roof” certificates, and enforce penalties for non‑compliance (₱5,000 per violation).  

### 3. SDG Alignment  

- **SDG 1 – No Poverty:** Micro‑finance grants and subsidies reduce the financial burden of safe‑roof upgrades, directly supporting low‑income households.  
- **SDG 11 – Sustainable Cities and Communities:** The Safe‑Roof and Flood‑Resilient ordinances promote resilient housing, inclusive urban planning, and reduced disaster loss.  
- **SDG 13 – Climate Action:** GreenRoof incentives lower carbon footprints and improve climate‑adaptive capacity by encouraging durable, energy‑efficient roofing materials.  

### 4. Impact  

| Indicator | Baseline | Target (3 yr) | Expected Change |
|-----------|----------|--------------|-----------------|
| % of households with vulnerable roofs | 89.8 % | ≤30 % | ↓ ≈ 60 pp |
| Number of “Safe‑Roof” certificates issued | 0 | 4,500 | ↑ 4,500 |
| Flood‑related damage cost (annual) | ₱12 M | ≤₱3 M | ↓ 75 % |
| Household access to micro‑finance | 0 | 1,200 families | ↑ 1,200 |

**Estimated Budget:** ₱150 M over three years  
- Barangay Internal Revenue Allocation (IRA): ₱45 M  
- NDRRMF & National Housing Authority grants: ₱70 M  
- Private sector CSR & NGOs: ₱20 M  
- Community contributions & in‑kind labor: ₱15 M  

**Funding Sources** are aligned with existing national programs, ensuring fiscal sustainability and compliance with the cited Republic Acts.

In [ ]:
import shutil

# Compress your chroma_db folder into a zip file
shutil.make_archive("chroma_db_backup", "zip", "chroma_db")

print("✅ Compression complete! Look for 'chroma_db_backup.zip' in your left sidebar files.")

✅ Compression complete! Look for 'chroma_db_backup.zip' in your left sidebar files.
